# Rebuild from scratch — Qwen3-8B+LoRA → parsefix → v35 symbolic approval layer

This notebook runs an end-to-end benchmark on two test files:

1. `generated_v4style_300.json` → expected flatten size ≈ 600 samples.
2. `benchmark_v2_1000.json` → expected flatten size ≈ 2000 samples.

It does **not** consume old `full_model_eval_v2/v3.2` prediction artifacts. It rebuilds predictions from the model, then applies:

- task-aware parsing,
- minimal parsefix for format contamination,
- v35 symbolic verifier, including E1: `∀x ¬Q(x) ⊨ ¬∃x Q(x)`,
- metrics, confusion matrices, certificate audit, and risk report.

Doctrine: **LLM proposes → parser fixes format → symbolic verifier approves proof-certified corrections → selector/report prevents artifact and overfit drift.**

In [ ]:
# ============================================================
# Config
# ============================================================
import os, re, json, glob, time, math, csv, gc
from pathlib import Path
from collections import Counter, defaultdict

try:
    import torch
except Exception as e:
    torch = None
    print('WARNING: torch import failed:', repr(e))

DATA_PATH = '/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json'
TEST_PATHS = {
    'generated_v4style_300': '/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json',
    'benchmark_v2_1000': '/kaggle/input/datasets/nguyenminhtric/test-pipeline/benchmark_v2_1000.json',
}
BASE_MODEL_PATH = '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'
ADAPTER_PATH = '/kaggle/input/datasets/yixuanisthebest/v2333333'

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to an integer for smoke test, None for full benchmark.
MAX_SAMPLES_PER_DATASET = None

# Inference settings. Keep short to prevent repeated Reasoning/Final Answer loops.
MAX_NEW_TOKENS = 160
DO_SAMPLE = False
TEMPERATURE = None
REPETITION_PENALTY = 1.05
CHECKPOINT_EVERY = 25

# Set False if you only want to re-score existing rebuild_*_preds.json files.
RUN_INFERENCE = True

LABELS7 = ['A','B','C','D','Yes','No','Unknown']
MC_LABELS = ['A','B','C','D']
YNU_LABELS = ['Yes','No','Unknown']

print('OUT_DIR =', OUT_DIR)
print('RUN_INFERENCE =', RUN_INFERENCE)
print('MAX_SAMPLES_PER_DATASET =', MAX_SAMPLES_PER_DATASET)


In [ ]:
# ============================================================
# Utility: load/save, flatten, prompt, parse, metrics
# ============================================================
def normalize_label(x):
    if x is None: return None
    s=str(x).strip()
    if not s: return None
    up=s.upper()
    if up in MC_LABELS: return up
    t=s.title()
    if t in YNU_LABELS: return t
    if up in ['YES','NO','UNKNOWN']:
        return {'YES':'Yes','NO':'No','UNKNOWN':'Unknown'}[up]
    return None

def load_json_path(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(obj, name):
    p = OUT_DIR / name
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    return str(p)

def save_csv(rows, name, fieldnames=None):
    p = OUT_DIR / name
    if fieldnames is None:
        keys=[]
        for r in rows:
            for k in r.keys():
                if k not in keys: keys.append(k)
        fieldnames=keys
    with open(p, 'w', encoding='utf-8', newline='') as f:
        w=csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows: w.writerow({k:r.get(k) for k in fieldnames})
    return str(p)

def as_list(x):
    if x is None: return []
    return x if isinstance(x, list) else [x]

def is_mc_question(q, ans=None):
    if normalize_label(ans) in MC_LABELS: return True
    q=str(q)
    return bool(re.search(r'(?m)^\s*A\s*[\.)]', q)) and bool(re.search(r'(?m)^\s*B\s*[\.)]', q))

def flatten_nested(raw, dataset_key):
    rows=[]
    assert isinstance(raw, list), f'{dataset_key}: expected list, got {type(raw).__name__}'
    for rec_i, rec in enumerate(raw):
        qs = rec.get('questions', []) if isinstance(rec, dict) else []
        ans = rec.get('answers', []) if isinstance(rec, dict) else []
        exps = rec.get('explanation', []) if isinstance(rec, dict) else []
        idxs = rec.get('idx', []) if isinstance(rec, dict) else []
        if not isinstance(qs, list): qs=[qs]
        if not isinstance(ans, list): ans=[ans]*len(qs)
        if not isinstance(exps, list): exps=[exps]*len(qs)
        if not isinstance(idxs, list): idxs=[idxs]*len(qs)
        for q_i, q in enumerate(qs):
            gold = ans[q_i] if q_i < len(ans) else None
            exp = exps[q_i] if q_i < len(exps) else None
            idx = idxs[q_i] if q_i < len(idxs) else None
            is_mc = is_mc_question(q, gold)
            rows.append({
                'dataset': dataset_key,
                'flat_id': f'{dataset_key}:{rec_i}:{q_i}',
                'rec_i': rec_i,
                'q_i': q_i,
                'idx': idx,
                'premises_FOL': as_list(rec.get('premises-FOL') or rec.get('premises_FOL')),
                'premises_NL': as_list(rec.get('premises-NL') or rec.get('premises_NL')),
                'question': str(q),
                'gold': normalize_label(gold),
                'gold_raw': gold,
                'reference_explanation': exp,
                'is_mc': bool(is_mc),
                'is_ynu': not bool(is_mc),
            })
    return rows

FINAL_RE = re.compile(r'Final\s*Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)
ANY_LABEL_RE = re.compile(r'\b(Yes|No|Unknown|[ABCD])\b', re.I)

def parse_answer(text, allowed=None):
    allowed = allowed or LABELS7
    if text is None: return 'UNPARSEABLE'
    s=str(text)
    hits = FINAL_RE.findall(s)
    labs=[normalize_label(h) for h in hits]
    labs=[x for x in labs if x in allowed]
    if labs: return labs[-1]
    tail='\n'.join(s.strip().splitlines()[-6:])
    hits=ANY_LABEL_RE.findall(tail)
    labs=[normalize_label(h) for h in hits]
    labs=[x for x in labs if x in allowed]
    return labs[-1] if labs else 'UNPARSEABLE'

def task_parse(text, is_mc):
    return parse_answer(text, MC_LABELS if is_mc else YNU_LABELS)

def build_prompt(row):
    prem_nl = row.get('premises_NL') or []
    prem_block = '\n'.join([f'{i+1}. {p}' for i,p in enumerate(prem_nl)])
    q = row['question']
    if row['is_mc']:
        final = 'A, B, C, or D'
        extra = 'Choose exactly one option. End with: Final Answer: <A/B/C/D>.'
    else:
        final = 'Yes, No, or Unknown'
        extra = 'Answer only the logical truth of the question from the premises. End with: Final Answer: <Yes/No/Unknown>.'
    return f"""You are solving a logic-based educational QA problem.

Premises:
{prem_block}

Question:
{q}

Instructions:
- Reason from the premises only.
- The final answer must be one of: {final}.
- {extra}
"""

def per_label_metrics(rows, pred_key='pred'):
    out={}
    for lab in LABELS7:
        tp=sum(1 for r in rows if r.get('gold')==lab and r.get(pred_key)==lab)
        fp=sum(1 for r in rows if r.get('gold')!=lab and r.get(pred_key)==lab)
        fn=sum(1 for r in rows if r.get('gold')==lab and r.get(pred_key)!=lab)
        support=sum(1 for r in rows if r.get('gold')==lab)
        pred_count=sum(1 for r in rows if r.get(pred_key)==lab)
        prec=tp/(tp+fp) if (tp+fp) else 0.0
        rec=tp/(tp+fn) if (tp+fn) else 0.0
        f1=2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        out[lab]=dict(precision=prec, recall=rec, f1=f1, support=support, pred_count=pred_count)
    return out

def metrics(rows, pred_key='pred'):
    n=len(rows)
    correct=sum(1 for r in rows if r.get('gold')==r.get(pred_key))
    pl=per_label_metrics(rows, pred_key)
    supports={lab:pl[lab]['support'] for lab in LABELS7}
    macro7=sum(pl[lab]['f1'] for lab in LABELS7)/7
    total_support=sum(supports.values()) or 1
    weighted=sum(pl[lab]['f1']*supports[lab] for lab in LABELS7)/total_support
    mc_macro=sum(pl[lab]['f1'] for lab in MC_LABELS)/4
    ynu_macro=sum(pl[lab]['f1'] for lab in YNU_LABELS)/3
    return dict(n=n, correct=correct, acc=correct/n if n else 0.0, micro_f1=correct/n if n else 0.0,
                macro7=macro7, weighted_f1=weighted, mc_macro=mc_macro, ynu_macro=ynu_macro,
                per_label=pl, pred_distribution=dict(Counter(r.get(pred_key) for r in rows)),
                gold_distribution=dict(Counter(r.get('gold') for r in rows)))

def confusion_rows(rows, pred_key='pred'):
    labels=LABELS7+['UNPARSEABLE']
    out=[]
    for g in labels:
        row={'gold':g}
        for p in labels:
            row[p]=sum(1 for r in rows if r.get('gold')==g and r.get(pred_key)==p)
        out.append(row)
    return out

print('Utilities ready.')


In [ ]:
# ============================================================
# v35 symbolic mini-engine
# ============================================================
def norm_fol(s):
    s=str(s)
    for a,b in [('∀','ForAll '),('∃','Exists '),('→','->'),('∧','&'),('∨','|'),('¬','~'),('↔','<->')]:
        s=s.replace(a,b)
    return s

ATOM=re.compile(r'(~?)\s*([A-Z][A-Za-z0-9_]*)\s*\(([^()]*)\)')
RES={'ForAll','Exists','Implies','And','Or','Not'}

def atoms_of(s):
    return [(n,neg=='~') for neg,n,_ in ATOM.findall(str(s)) if n not in RES]

def build_kb(pfs):
    kb={'edges':[],'ufacts':set(),'efacts':set(),'neg_ufacts':set(),'raw':list(pfs or [])}
    for raw in pfs or []:
        s=norm_fol(raw)
        ats=atoms_of(s)
        if '->' in s and len(ats)>=2:
            ant, cons = ats[0], ats[-1]
            kb['edges'].append((ant[0], cons[0], cons[1], raw))
        elif ats:
            name,neg=ats[0]
            if 'ForAll' in s:
                (kb['neg_ufacts'] if neg else kb['ufacts']).add(name)
            elif 'Exists' in s:
                kb['efacts'].add((name,neg))
    return kb

def close_universal(kb):
    pos=set(kb['ufacts']); neg=set(kb['neg_ufacts']); changed=True; paths={p:f'∀x {p}(x)' for p in pos}
    neg_paths={p:f'∀x ¬{p}(x)' for p in neg}
    while changed:
        changed=False
        for a,b,bneg,raw in kb['edges']:
            if a in pos:
                if bneg and b not in neg:
                    neg.add(b); neg_paths[b]=f'{paths.get(a,a)} + {raw}'; changed=True
                if not bneg and b not in pos:
                    pos.add(b); paths[b]=f'{paths.get(a,a)} + {raw}'; changed=True
            # contrapositive for single-antecedent universal implications: A -> B gives ~B -> ~A
            if not bneg and b in neg and a not in neg:
                neg.add(a); neg_paths[a]=f'CONTRAPOSITIVE_OF: {raw}'; changed=True
    return pos,neg,paths,neg_paths

def tokenize_words(s):
    return set(re.findall(r'[a-z]+', str(s).lower()))

def pred_to_words(p):
    return set(w.lower() for w in re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', str(p)))

DOMAIN_WORDS={'student','students','intern','interns','course','courses','researcher','researchers','developer','developers','trainee','trainees','employee','employees','teacher','teachers','athlete','athletes','applicant','applicants','candidate','candidates','participant','participants','member','members','librarian','librarians','patron','patrons','mentor','mentors','club','lab','workshop','scholarship'}
STOP_TARGET={'all','some','at','least','one','every','is','are','does','do','according','premises','true','statement','can','we','conclude','based','following','there','who','that','the','a','an','if','then'} | DOMAIN_WORDS

def qtype(q):
    ql=str(q).lower()
    if re.search(r'\b(at least one|there is at least one|there exists|some)\b', ql): return 'existential'
    if re.search(r'\b(all|every)\b', ql): return 'universal'
    return 'atomic'

def choose_target(q, predicates):
    qw=tokenize_words(q)-STOP_TARGET
    best=(0.0,None,set())
    for p in predicates:
        pw=pred_to_words(p)-STOP_TARGET
        if not pw: continue
        inter=len(qw & pw)
        score=inter/max(1,len(pw))
        if score>best[0]: best=(score,p,pw)
    return best[1], best[0], sorted(best[2])

def v35_verdict(question, kb):
    pos,neg,pos_paths,neg_paths=close_universal(kb)
    preds=set(pos)|set(neg)|{a for a,_,_,_ in kb['edges']}|{b for _,b,_,_ in kb['edges']}
    target,score,tw=choose_target(question, preds)
    qt=qtype(question)
    cert=dict(qtype=qt,target=target,target_score=score,target_words=tw,
              neg_proof_universal=bool(target in neg) if target else False,
              pos_proof_universal=bool(target in pos) if target else False,
              neg_proof_path=neg_paths.get(target) if target else None,
              pos_proof_path=pos_paths.get(target) if target else None)
    if not target or score < 0.50:
        cert['reason']='NO_TARGET_MATCH'
        return 'ABSTAIN', cert
    # E1: if no entity can be Q, existential Q is false. This intentionally overrides positive conflict.
    if qt=='existential' and target in neg:
        cert['rule']='E1 forall-neg entails not-exists'
        return 'PROVE_NO', cert
    # Conservative PROVE_NO for universal/atomic: abstain if positive proof also exists.
    if target in neg:
        if target in pos:
            cert['reason']=f'POSITIVE-PROOF CONFLICT ({qt})'
            return 'ABSTAIN', cert
        cert['rule']='typed PROVE_NO no-positive-conflict'
        return 'PROVE_NO', cert
    cert['reason']='NO_NEGATIVE_PROOF'
    return 'ABSTAIN', cert

print('v35 symbolic engine ready.')


In [ ]:
# ============================================================
# Model loading
# ============================================================
model = None
tok = None

def load_model_once():
    global model, tok
    if model is not None and tok is not None:
        return model, tok
    if not RUN_INFERENCE:
        print('RUN_INFERENCE=False; model not loaded.')
        return None, None
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    assert torch is not None, 'torch is required for inference'
    print('Loading tokenizer:', ADAPTER_PATH)
    tok = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True, local_files_only=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    kw = dict(trust_remote_code=True, local_files_only=True, torch_dtype=dtype)
    if torch.cuda.is_available():
        kw['device_map']='auto'
    print('Loading base model:', BASE_MODEL_PATH, 'dtype=', dtype, 'cuda=', torch.cuda.is_available())
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH, **kw)
    print('Loading LoRA adapter:', ADAPTER_PATH)
    model = PeftModel.from_pretrained(base, ADAPTER_PATH, local_files_only=True)
    model.eval()
    return model, tok

def render_chat_prompt(prompt):
    if tok is not None and getattr(tok, 'chat_template', None):
        messages=[{'role':'user','content':prompt}]
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

@torch.inference_mode() if torch is not None else (lambda f:f)
def generate_one(prompt):
    m,t = load_model_once()
    rendered = render_chat_prompt(prompt)
    inputs = t(rendered, return_tensors='pt')
    if torch.cuda.is_available():
        dev = next(m.parameters()).device
        inputs = {k:v.to(dev) for k,v in inputs.items()}
    gen_kw=dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE, repetition_penalty=REPETITION_PENALTY,
                pad_token_id=t.pad_token_id, eos_token_id=t.eos_token_id)
    if TEMPERATURE is not None:
        gen_kw['temperature']=TEMPERATURE
    out = m.generate(**inputs, **gen_kw)
    new_ids = out[0][inputs['input_ids'].shape[-1]:]
    text = t.decode(new_ids, skip_special_tokens=True)
    return text

print('Model loader ready.')


In [ ]:
# ============================================================
# Inference + parsefix + v35 for one dataset
# ============================================================
def apply_parsefix(row):
    return task_parse(row.get('raw_output',''), row['is_mc'])

def apply_v35(row, pred_key='pred_parsefix'):
    pred = row.get(pred_key)
    cert=None; verdict='ABSTAIN'
    newpred=pred
    if row.get('is_ynu') and pred!='No':
        verdict, cert = v35_verdict(row['question'], build_kb(row.get('premises_FOL') or []))
        if verdict == 'PROVE_NO':
            newpred='No'
    return newpred, verdict, cert

def run_dataset(dataset_key, test_path):
    print('\n'+'='*80)
    print('DATASET:', dataset_key)
    print('PATH:', test_path)
    raw = load_json_path(test_path)
    rows = flatten_nested(raw, dataset_key)
    if MAX_SAMPLES_PER_DATASET is not None:
        rows = rows[:MAX_SAMPLES_PER_DATASET]
    print('flattened samples:', len(rows), 'gold:', Counter(r['gold'] for r in rows), 'is_mc:', sum(r['is_mc'] for r in rows))
    out_pred_name=f'rebuild_{dataset_key}_preds.json'
    out_path=OUT_DIR/out_pred_name
    existing=[]
    if out_path.exists():
        try:
            existing=json.load(open(out_path, encoding='utf-8'))
            print('Found existing checkpoint:', out_path, 'rows=', len(existing))
        except Exception as e:
            print('Could not read existing checkpoint:', repr(e))
            existing=[]
    by_id={r.get('flat_id'):r for r in existing if r.get('raw_output') is not None}
    results=[]
    t0=time.time()
    if RUN_INFERENCE:
        load_model_once()
    for i,row in enumerate(rows):
        rid=row['flat_id']
        if rid in by_id:
            nr=dict(row); nr.update({k:v for k,v in by_id[rid].items() if k not in ('premises_FOL','premises_NL','question','gold','is_mc','is_ynu')})
            results.append(nr)
            continue
        nr=dict(row)
        prompt=build_prompt(nr)
        start=time.time()
        if RUN_INFERENCE:
            try:
                raw_out=generate_one(prompt)
                err=None
            except Exception as e:
                raw_out=''
                err=repr(e)
                print('GEN ERROR', rid, err)
        else:
            raw_out=nr.get('raw_output','')
            err='RUN_INFERENCE_FALSE_NO_EXISTING_OUTPUT'
        latency=time.time()-start
        nr['prompt_preview']=prompt[:800]
        nr['raw_output']=raw_out
        nr['generation_error']=err
        nr['latency_sec']=latency
        nr['pred_raw']=task_parse(raw_out, nr['is_mc'])
        results.append(nr)
        if (i+1)%CHECKPOINT_EVERY==0:
            save_json(results, out_pred_name)
            print(f'checkpoint {dataset_key}: {i+1}/{len(rows)} elapsed={(time.time()-t0)/60:.1f}m')
    for r in results:
        if 'pred_raw' not in r:
            r['pred_raw']=task_parse(r.get('raw_output',''), r['is_mc'])
        r['pred_parsefix']=apply_parsefix(r)
        pred35, verdict, cert = apply_v35(r, 'pred_parsefix')
        r['pred_v35']=pred35
        r['v35_verdict']=verdict
        r['v35_certificate']=cert
        r['v35_changed']=pred35 != r['pred_parsefix']
        if r['v35_changed']:
            r['v35_posthoc']='GOOD_FLIP' if r.get('gold')==pred35 else 'BAD_FLIP'
    pred_path=save_json(results, out_pred_name)
    reloaded=json.load(open(pred_path, encoding='utf-8'))
    assert len(reloaded)==len(results), 'save/reload row count mismatch'
    m_raw=metrics(reloaded,'pred_raw')
    m_pf=metrics(reloaded,'pred_parsefix')
    m35=metrics(reloaded,'pred_v35')
    flips=[dict(flat_id=r['flat_id'], old=r['pred_parsefix'], new=r['pred_v35'], gold=r['gold'],
                question=r['question'], certificate=r.get('v35_certificate'), posthoc=r.get('v35_posthoc')) for r in reloaded if r.get('v35_changed')]
    good=sum(1 for f in flips if f['posthoc']=='GOOD_FLIP')
    bad=sum(1 for f in flips if f['posthoc']=='BAD_FLIP')
    summary=dict(
        version='rebuild_from_scratch_v35', dataset=dataset_key, input_path=test_path,
        n=len(reloaded), n_mc=sum(r['is_mc'] for r in reloaded), n_ynu=sum(r['is_ynu'] for r in reloaded),
        metrics_raw=m_raw, metrics_parsefix=m_pf, metrics_v35=m35,
        v35_flips=flips, n_v35_flips=len(flips), posthoc_good=good, posthoc_bad=bad,
        latency=dict(total_sec=sum(float(r.get('latency_sec') or 0) for r in reloaded),
                     mean_sec=sum(float(r.get('latency_sec') or 0) for r in reloaded)/max(1,len(reloaded))),
        invalid_counts=dict(raw=sum(r.get('pred_raw')=='UNPARSEABLE' for r in reloaded),
                            parsefix=sum(r.get('pred_parsefix')=='UNPARSEABLE' for r in reloaded),
                            v35=sum(r.get('pred_v35')=='UNPARSEABLE' for r in reloaded)),
        output_pred_file=os.path.basename(pred_path),
    )
    summary_path=save_json(summary, f'rebuild_{dataset_key}_summary.json')
    conf_path=save_csv(confusion_rows(reloaded,'pred_v35'), f'rebuild_{dataset_key}_confusion_v35.csv')
    errors=[r for r in reloaded if r.get('gold')!=r.get('pred_v35')]
    err_path=save_json(errors[:300], f'rebuild_{dataset_key}_error_cases_preview.json')
    print(f'{dataset_key}: raw acc={m_raw["acc"]:.4f}, parsefix acc={m_pf["acc"]:.4f}, v35 acc={m35["acc"]:.4f}')
    print(f'{dataset_key}: raw macro7={m_raw["macro7"]:.4f}, parsefix macro7={m_pf["macro7"]:.4f}, v35 macro7={m35["macro7"]:.4f}')
    print(f'{dataset_key}: v35 flips={len(flips)} good={good} bad={bad}')
    print('saved:', pred_path, summary_path, conf_path, err_path)
    return summary

print('Dataset runner ready.')


In [ ]:
# ============================================================
# Run both test sets
# ============================================================
all_summaries={}
for key,path in TEST_PATHS.items():
    if not Path(path).exists():
        print('WARNING: missing test path:', path)
        continue
    all_summaries[key]=run_dataset(key, path)

risk_report=dict(
    version='rebuild_from_scratch_v35_risk_report',
    artifact_risk='LOW_RELOADED_SAVED_PREDS_WITH_CHECKPOINTS',
    overfit_risk='LOW_SYMBOLIC_RULES_NO_SWEEP_APPLIED_AFTER_MODEL_INFERENCE',
    underfit_risk='REMAINING: MC option-proof not applied; statement-form YNU limited; predicate matching imperfect; generation quality still matters',
    label_collapse=False,
    notes=[
        'This notebook rebuilds predictions from model inference; it does not consume old v2/v3.2 prediction artifacts.',
        'v35 only applies proof-certified No corrections; MC symbolic proof remains disabled.',
        'Use benchmark_v2_1000 issue report to separate direct-conflict/canary subsets if needed.',
    ]
)
combined=dict(version='rebuild_from_scratch_two_tests_v35', summaries=all_summaries, risk_report=risk_report)
combined_path=save_json(combined,'rebuild_combined_summary.json')
risk_path=save_json(risk_report,'rebuild_v35_risk_report.json')
print('COMBINED SAVED:', combined_path, risk_path)


## Expected outputs

For each dataset key (`generated_v4style_300`, `benchmark_v2_1000`), the notebook writes:

- `rebuild_<dataset>_preds.json`
- `rebuild_<dataset>_summary.json`
- `rebuild_<dataset>_confusion_v35.csv`
- `rebuild_<dataset>_error_cases_preview.json`

And combined:

- `rebuild_combined_summary.json`
- `rebuild_v35_risk_report.json`

The notebook can resume from existing `rebuild_<dataset>_preds.json` checkpoints.